## Cafe Sales - Dirty Data for Cleaning Training
[Dirty Cafe Sales Dataset](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training)
### Overview
_The Dirty Cafe Sales dataset contains 10,000 rows of synthetic data representing sales transactions in a cafe. This dataset is intentionally "dirty," with missing values, inconsistent data, and errors introduced to provide a realistic scenario for data cleaning and exploratory data analysis (EDA). It can be used to practice cleaning techniques, data wrangling, and feature engineering._

`File Information: `<br>
* File Name: dirty_cafe_sales.csv
* Number of Rows: 10,000
* Number of Columns: 8

In [13]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [15]:
df.dtypes

Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

_Tipificando as colunas para os tipos corretos:_

In [16]:
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Quantity'] = df['Quantity'].astype('Int64', errors='ignore')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
df.dtypes

Transaction ID              object
Item                        object
Quantity                     Int64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

In [17]:
df.describe()

,Quantity,Price Per Unit,Total Spent,Transaction Date
count,9521.0,9467.000000,9498.000000,9540
mean,3.028463,2.949984,8.924352,2023-07-01 23:00:31.698113280
min,1.0,1.000000,1.000000,2023-01-01 00:00:00
25%,2.0,2.000000,4.000000,2023-04-01 00:00:00
50%,3.0,3.000000,8.000000,2023-07-02 00:00:00
75%,4.0,4.000000,12.000000,2023-10-02 00:00:00
max,5.0,5.000000,25.000000,2023-12-31 00:00:00
std,1.419007,1.278450,6.009919,NaN


In [18]:
df.duplicated().sum()

np.int64(0)

In [19]:
df['Transaction ID'].nunique()

10000

In [20]:
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      2579
Location            3265
Transaction Date     460
dtype: int64

In [21]:
df['Item'].value_counts()

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
ERROR        292
Name: count, dtype: int64

In [22]:
df['Price Per Unit'].value_counts()

Price Per Unit
3.0    2429
4.0    2331
2.0    1227
5.0    1204
1.0    1143
1.5    1133
Name: count, dtype: int64

**Tratando valores Faltantes:**

_Identifiquei dependência funcional parcial entre Item e Price Per Unit, e usei isso para imputação baseada em regras de negócio, evitando métodos estatísticos simples que poderiam introduzir viés._

In [23]:
# df[df['Price Per Unit'] == '1.0'].tail(20) → Cookie 
# df[df['Price Per Unit'] == '1.5'].tail(20) → Tea
# df[df['Price Per Unit'] == '2.0'].tail(20) → Coffee
# df[df['Price Per Unit'] == '3.0'].tail(20) → Juice/Cake
# df[df['Price Per Unit'] == '4.0'].tail(20) → Smoothie/Sandwich
# df[df['Price Per Unit'] == '5.0'].tail(20) → Salad

In [24]:
df['Item'] = df['Item'].replace({'UNKNOWN': None, 'ERROR': None})

In [25]:
print(df['Item'].isnull().sum())
print(df['Price Per Unit'].isnull().sum())

969
533


In [26]:
item_to_price = {
  'Cookie': 1.0,
  'Tea': 1.5,
  'Coffee': 2.0,
  'Salad': 5.0,
  'Juice': 3.0,
  'Cake': 3.0,
  'Smoothie': 4.0,
  'Sandwich': 4.0
}

price_to_item = {
  1.0: 'Cookie',
  1.5: 'Tea',
  2.0: 'Coffee',
  5.0: 'Salad'
}
  
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].map(item_to_price))
df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(price_to_item))

**Data Quality Check baseado em regras de negócio:**

In [27]:
#Verificando se existem inconsistências:
#((df['Price Per Unit'] == 1.0) & (df['Item'] != 'Cookie')).value_counts()
#((df['Price Per Unit'] == 1.5) & (df['Item'] != 'Tea')).value_counts()
#((df['Price Per Unit'] == 2.0) & (df['Item'] != 'Coffee')).value_counts()
#((df['Price Per Unit'] == 5.0) & (df['Item'] != 'Salad')).value_counts()

_Casos ambíguos (preços associados a múltiplos itens) foram categorizados como ambíguos, evitando imputações arbitrárias que poderiam comprometer a integridade analítica dos dados:_

In [28]:
df.loc[df['Item'].isna() & (df['Price Per Unit'] == 3.0), 'Item'] = 'Unknown_3.0'
df.loc[df['Item'].isna() & (df['Price Per Unit'] == 4.0), 'Item'] = 'Unknown_4.0'

In [29]:
print(df['Item'].isnull().sum())
print(df['Price Per Unit'].isnull().sum())

54
54


In [30]:
df[df['Item'].isna() | df['Price Per Unit'].isna()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
118,TXN_4633784,NaN,5,NaN,15.0,NaN,In-store,2023-02-06
151,TXN_4031509,NaN,4,NaN,16.0,Credit Card,Takeaway,2023-01-04
289,TXN_3495950,NaN,4,NaN,6.0,Credit Card,In-store,2023-02-19
334,TXN_2523298,NaN,4,NaN,6.0,ERROR,In-store,2023-03-25
550,TXN_4186681,NaN,4,NaN,6.0,Digital Wallet,NaN,2023-05-24
750,TXN_5787508,NaN,3,NaN,9.0,Credit Card,Takeaway,2023-07-23
818,TXN_7940202,NaN,1,NaN,4.0,Digital Wallet,NaN,2023-07-23
1154,TXN_2473090,NaN,2,NaN,3.0,Credit Card,In-store,2023-03-03
1337,TXN_5031214,NaN,5,NaN,5.0,NaN,Takeaway,2023-07-29
1377,TXN_8396271,NaN,2,NaN,2.0,NaN,NaN,2023-09-12


_Após imputações baseadas em regras de negócio e tratamento explícito de ambiguidade, restaram menos de 1% de registros com ausência crítica de dados, os quais foram removidos por inviabilidade de reconstrução confiável:_

In [31]:
df = df.dropna(subset=['Item', 'Price Per Unit'])

In [32]:
df.isnull().sum()

Transaction ID         0
Item                   0
Quantity             476
Price Per Unit         0
Total Spent          499
Payment Method      2562
Location            3242
Transaction Date     457
dtype: int64

**Reconstrução de dados baseada em dependência funcional :**

In [33]:
df['Quantity'] = df['Quantity'].apply(lambda x: x if pd.isna(x) else round(x))
df['Total Spent'] = df['Total Spent'].fillna(df['Price Per Unit'] * df['Quantity'])
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])

In [34]:
print(df['Quantity'].isnull().sum())
print(df['Total Spent'].isnull().sum())

20
20


In [35]:
# Verificando se existem inconsistências entre Total Spent, Price Per Unit e Quantity:
(abs(df['Total Spent'] - df['Price Per Unit'] * df['Quantity']) > 0.01).sum()

np.int64(0)

_Registros com ausência simultânea de Quantity e Total Spent foram considerados irrecuperáveis, sendo removidos por falta de informação suficiente para reconstrução confiável:_

In [36]:
df[df['Quantity'].isna() | df['Total Spent'].isna()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
236,TXN_8562645,Salad,NaN,5.0,NaN,NaN,In-store,2023-05-18
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
641,TXN_2962976,Juice,NaN,3.0,NaN,NaN,NaN,2023-03-17
738,TXN_8696094,Sandwich,NaN,4.0,NaN,NaN,Takeaway,2023-05-14
2796,TXN_9188692,Cake,NaN,3.0,NaN,Credit Card,NaN,2023-12-01
3203,TXN_4565754,Smoothie,NaN,4.0,NaN,Digital Wallet,Takeaway,2023-10-06
3224,TXN_6297232,Coffee,NaN,2.0,NaN,NaN,NaN,2023-04-07
3401,TXN_3251829,Tea,NaN,1.5,NaN,Digital Wallet,In-store,2023-07-25
4257,TXN_6470865,Coffee,NaN,2.0,NaN,Digital Wallet,Takeaway,2023-09-18
5841,TXN_5884081,Cookie,NaN,1.0,NaN,Digital Wallet,In-store,2023-07-05


In [37]:
df = df.dropna(subset=['Quantity', 'Total Spent'])

In [38]:
df.isnull().sum()

Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      2555
Location            3237
Transaction Date     457
dtype: int64

In [39]:
df['Location'].value_counts()

Location
Takeaway    3002
In-store    2994
ERROR        356
UNKNOWN      337
Name: count, dtype: int64

In [40]:
df['Payment Method'].value_counts()

Payment Method
Digital Wallet    2277
Credit Card       2253
Cash              2246
ERROR              303
UNKNOWN            292
Name: count, dtype: int64

**Category normalization + missing value standardization**

_Valores inconsistentes como 'ERROR' e 'UNKNOWN' foram padronizados para uma única categoria 'Unknown', garantindo consistência semântica e evitando duplicidade de categorias._

In [41]:
df['Payment Method'] = df['Payment Method'].replace(['ERROR', 'UNKNOWN'], None)
df['Location'] = df['Location'].replace(['ERROR', 'UNKNOWN'], None)

_Campos categóricos sem possibilidade de inferência foram preenchidos com categoria 'Unknown', enquanto registros com ausência de informação temporal foram removidos devido ao impacto em análises temporais :_

In [42]:
df['Payment Method'] = df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')
df = df.dropna(subset=['Transaction Date'])
df.isnull().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

In [43]:
df['Location'].value_counts()

Location
Unknown     3750
Takeaway    2869
In-store    2850
Name: count, dtype: int64

In [44]:
df['Payment Method'].value_counts()

Payment Method
Unknown           2987
Digital Wallet    2183
Credit Card       2151
Cash              2148
Name: count, dtype: int64

**Padronização final**<br>
*Nomes de colunas consistentes (snake_case)*

In [45]:
df.columns = df.columns.str.replace(' ', '_', regex=True).str.lower()
df.head()

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [ ]:
df.to_csv('data/clean_cafe_sales.csv', index=False)